# STT — 01 · Exploration des données (EDA)

Datasets (dans `data/`, non versionnés) :
- **fr-FR** et **fr-CA** : Common Voice Scripted Speech 26.0 (segments), découpage officiel train / dev / test ;
- **Spontané** : Common Voice Spontaneous Speech 5.0 — French.

Chaque point de l'EDA doit aboutir à une **décision** pour le modèle.

## Point 1 — Transcriptions manquantes (fr-FR, train)

Objectif : vérifier que chaque clip a bien un texte (`sentence`), sinon il n'a pas de cible pour la loss.

In [ ]:
import pandas as pd

# Lecture du fichier de découpage "train" de fr-FR.
# Les colonnes d'un .tsv sont séparées par des tabulations, d'où sep="\t".
train_fr = pd.read_csv("data/fr-FR/train.tsv", sep="\t")

# Forme du tableau : (nombre de clips, nombre de colonnes)
print("Forme du tableau :", train_fr.shape)

# isna() marque les valeurs manquantes (True), sum() les compte colonne par colonne
print("Valeurs manquantes par colonne :")
print(train_fr.isna().sum())

# Aperçu des 5 premiers clips (affiché automatiquement en dernière ligne de cellule)
train_fr.head()

**Résultat (fr-FR train)** : `sentence` n'a aucune valeur manquante. `age`, `gender` et `variant` sont souvent vides (champs facultatifs du profil Common Voice) ; `sentence_domain` et `segment` sont presque entièrement vides.

**Décision** : pour l'entraînement, on ne garde que `path` (audio), `sentence` (cible) et `client_id` (locuteur). Les autres colonnes servent seulement à décrire les données.

## Point 1 (suite) — fr-CA et dataset spontané

In [2]:
# fr-CA : même structure que fr-FR, le texte est dans la colonne "sentence"
train_ca = pd.read_csv("data/fr-CA/train.tsv", sep="\t")
# On ne compte les valeurs manquantes que dans la colonne utile
print("fr-CA train — phrases manquantes :", train_ca["sentence"].isna().sum())

# Spontané : structure différente, le texte est dans la colonne "transcription"
spont = pd.read_csv("data/sps-corpus-5.0-2026-09-11-fr/ss-corpus-fr.tsv", sep="\t")
print("Spontané — forme :", spont.shape)
print("Spontané — transcriptions manquantes :", spont["transcription"].isna().sum())

# Valeurs de la colonne "split" et leur nombre.
# dropna=False : compter aussi les valeurs manquantes (ignorées par défaut)
print(spont["split"].value_counts(dropna=False))

fr-CA train — phrases manquantes : 0
Spontané — forme : (688, 19)
Spontané — transcriptions manquantes : 394
split
NaN    688
Name: count, dtype: int64


**Résultat** : fr-CA n'a aucune phrase manquante. Spontané : **394 / 688 clips sans transcription** (57 %), et la colonne `split` est entièrement vide (pas de découpage officiel : à faire nous-mêmes, par locuteur).

### Combien de données spontanées utilisables ?

In [3]:
# On ne garde que les clips du spontané qui ont une transcription
spont_ok = spont[spont["transcription"].notna()]
print("Clips spontanés transcrits :", len(spont_ok))

# Nombre de locuteurs différents parmi ces clips
print("Locuteurs différents :", spont_ok["client_id"].nunique())

# Durée totale, en minutes (duration_ms est en millisecondes)
print("Durée totale :", spont_ok["duration_ms"].sum() / 1000 / 60, "minutes")

Clips spontanés transcrits : 294
Locuteurs différents : 41
Durée totale : 65.187 minutes


**Décision** : tout le spontané (294 clips transcrits, 41 locuteurs, 65 min) sert de **jeu de test** sur parole naturelle.

## Point 2 — Inventaire des caractères

Objectif : connaître tous les caractères présents dans les transcriptions, car ils définissent la sortie du modèle (un score par caractère + le blank du CTC).

In [4]:
from collections import Counter

# On rassemble toutes les phrases d'entraînement des deux segments
all_sentences = pd.concat([train_fr["sentence"], train_ca["sentence"]])

# On compte chaque caractère dans l'ensemble des phrases
char_counts = Counter("".join(all_sentences))

print("Nombre de caractères différents :", len(char_counts))
# Affichage du plus fréquent au plus rare
for char, count in char_counts.most_common():
    print(repr(char), count)

Nombre de caractères différents : 178
' ' 116045
'e' 95777
's' 52154
'a' 49418
't' 48173
'n' 47602
'i' 45825
'r' 44181
'l' 36976
'u' 35687
'o' 34469
'd' 24273
'c' 20150
'p' 17766
'm' 17067
'é' 16047
'.' 12605
'v' 8309
'g' 7348
'f' 6424
"'" 6030
'h' 5637
'b' 5482
',' 5003
'q' 4594
'L' 3776
'è' 2999
'x' 2726
'à' 2576
'I' 2494
'C' 2457
'y' 2166
'-' 2127
'j' 1717
'S' 1594
'’' 1470
'E' 1436
'A' 1314
'M' 1212
'P' 1169
'D' 992
'z' 838
'ê' 830
'B' 806
'k' 720
'T' 684
'J' 655
'R' 613
'G' 546
'U' 533
'O' 524
'F' 523
'N' 471
'ç' 436
'V' 416
'H' 385
'â' 342
'ô' 329
'î' 321
'!' 318
'w' 310
'É' 275
'?' 252
'K' 231
'́' 187
'œ' 181
'W' 174
'Q' 168
'À' 138
'ï' 105
'û' 94
'̀' 88
'Y' 80
'ù' 76
'…' 55
';' 51
'Z' 46
'á' 41
'̂' 39
':' 38
'í' 25
'ë' 23
'—' 23
'ó' 22
'X' 22
'ü' 17
'Ç' 17
'ö' 17
'š' 14
'ō' 13
'Â' 10
'ā' 9
'ć' 8
'«' 7
'»' 7
'ä' 7
'&' 6
'ñ' 6
'č' 6
'̧' 5
'ł' 5
'æ' 5
'ı' 5
'Î' 4
'ú' 4
')' 4
'´' 4
'–' 4
'ø' 4
'"' 4
'Š' 3
'ÿ' 3
'(' 3
'Ê' 3
'ạ' 3
'Á' 3
'Ō' 3
'ū' 3
'Č' 3
'ī' 3
'ă' 3
'ş' 3
'ð' 3
'È' 3

**Résultat** : 178 caractères différents (majuscules, ponctuation, deux apostrophes, accents décomposés, lettres étrangères, symboles vus une seule fois). Aucun chiffre.

**Décisions** : minuscules ; ponctuation gardée `. , ? !` + `'` et `-` ; ponctuation secondaire remplacée par un espace ; phrases contenant des caractères rares/étrangers écartées.

### Normalisation du texte

In [5]:
import re
import unicodedata

# Caractères autorisés : lettres françaises + espace + ponctuation retenue
ALLOWED = set("abcdefghijklmnopqrstuvwxyz" "àâæçéèêëîïôœùûüÿ" " '-.,?!")

def normalize(text):
    # Recolle les accents décomposés ("e" + "´" -> "é")
    text = unicodedata.normalize("NFC", text)
    # Tout en minuscules
    text = text.lower()
    # Toutes les variantes d'apostrophe deviennent ' (la même pour tout le monde)
    text = re.sub(r"[’‘ʼ´]", "'", text)
    # Ponctuation secondaire remplacée par un espace
    text = re.sub(r"[;:…«»()\"—–]", " ", text)
    # Plusieurs espaces d'affilée -> un seul ; espaces de début et fin supprimés
    text = re.sub(r" +", " ", text).strip()
    return text

def is_valid(text):
    # Vrai si tous les caractères du texte sont autorisés
    return set(text) <= ALLOWED

# Application à toutes les phrases d'entraînement
normalized = all_sentences.map(normalize)
valid = normalized.map(is_valid)

print("Phrases au total :", len(normalized))
print("Phrases écartées :", (~valid).sum())

# Nouvel inventaire, uniquement sur les phrases gardées
new_counts = Counter("".join(normalized[valid]))
print("Caractères après normalisation :", len(new_counts))
print(sorted(new_counts))

Phrases au total : 13163
Phrases écartées : 181
Caractères après normalisation : 49
[' ', '!', "'", ',', '-', '.', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'à', 'â', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'î', 'ï', 'ô', 'ù', 'û', 'ü', 'ÿ', 'œ']


**Résultat** : 181 phrases écartées sur 13 163 (1,4 %) ; **49 caractères** restants → l'output head aura **50 sorties** (49 + blank).

## Points 3 et 4 — Durée des clips et fréquence d'échantillonnage

Lecture des mp3 avec **soundfile** (choix : torchaudio nécessiterait FFmpeg « shared » sur Windows ; torchaudio servira au log-Mel et au rééchantillonnage).

In [6]:
import soundfile as sf

def add_audio_info(df, folder):
    # Pour chaque clip, lit seulement l'en-tête du fichier mp3
    # (pas tout le son) : c'est rapide
    durations = []
    sample_rates = []
    for path in df["path"]:
        info = sf.info(f"{folder}/clips/{path}")
        durations.append(info.duration)        # durée en secondes
        sample_rates.append(info.samplerate)   # échantillons par seconde
    # On ajoute deux nouvelles colonnes au tableau
    df["duration"] = durations
    df["sample_rate"] = sample_rates

add_audio_info(train_fr, "data/fr-FR")
add_audio_info(train_ca, "data/fr-CA")

for name, df in [("fr-FR", train_fr), ("fr-CA", train_ca)]:
    print(f"=== {name}")
    print("Durée totale :", round(df["duration"].sum() / 3600, 2), "heures")
    print("Fréquences d'échantillonnage :")
    print(df["sample_rate"].value_counts())
    print("Statistiques des durées (secondes) :")
    print(df["duration"].describe())

=== fr-FR
Durée totale : 7.12 heures
Fréquences d'échantillonnage :
sample_rate
32000    3911
48000    1161
44100      14
Name: count, dtype: int64
Statistiques des durées (secondes) :
count    5086.000000
mean        5.043195
std         1.364073
min         1.699500
25%         4.039500
50%         4.949625
75%         5.875500
max        10.627500
Name: duration, dtype: float64
=== fr-CA
Durée totale : 11.81 heures
Fréquences d'échantillonnage :
sample_rate
48000    4413
32000    3664
Name: count, dtype: int64
Statistiques des durées (secondes) :
count    8077.000000
mean        5.262463
std         1.484478
min         1.469625
25%         4.199656
50%         5.189625
75%         6.235500
max        11.239500
Name: duration, dtype: float64


**Résultat** : 18,93 h d'entraînement (fr-FR 7,12 h + fr-CA 11,81 h) ; clips de 1,5 à 11,2 s (médiane ≈ 5 s) ; **3 fréquences d'échantillonnage** (32 000, 44 100, 48 000 Hz).

**Décisions** : tout rééchantillonner à **16 000 Hz** (standard parole, Nyquist 8 kHz, données 2 à 3× plus légères) ; garder **tous les clips** (durées homogènes).

## Point 5 — Contrainte du CTC

Le CTC exige : **nombre d'instants ≥ nombre de caractères + nombre de lettres doublées** (un blank obligatoire entre deux lettres identiques).
On mesure combien d'instants par seconde nos phrases demandent, pour savoir jusqu'où le modèle pourra réduire le nombre d'instants (sous-échantillonnage temporel).

In [7]:
def count_repeats(text):
    # Compte les paires de caractères identiques qui se suivent ("nn", "ss", "ll"...)
    # zip(text, text[1:]) associe chaque caractère au suivant
    return sum(a == b for a, b in zip(text, text[1:]))

for name, df in [("fr-FR", train_fr), ("fr-CA", train_ca)]:
    # Texte normalisé (la fonction validée au point 2)
    df["text"] = df["sentence"].map(normalize)
    # Instants minimum nécessaires pour le CTC : caractères + blanks obligatoires
    df["ctc_min"] = df["text"].str.len() + df["text"].map(count_repeats)
    # Instants nécessaires par seconde d'audio
    df["ctc_per_sec"] = df["ctc_min"] / df["duration"]

    print(f"=== {name} — instants CTC nécessaires par seconde")
    print(df["ctc_per_sec"].describe())
    # Proportion de clips qui ne tiendraient pas avec une réduction ÷4 (25/s) ou ÷8 (12,5/s)
    print("Clips impossibles à 25 instants/s :", (df["ctc_per_sec"] > 25).sum())
    print("Clips impossibles à 12,5 instants/s :", (df["ctc_per_sec"] > 12.5).sum())

=== fr-FR — instants CTC nécessaires par seconde
count    5086.000000
mean       12.594744
std         2.693740
min         2.930898
25%        10.827492
50%        12.687615
75%        14.440156
max        21.805864
Name: ctc_per_sec, dtype: float64
Clips impossibles à 25 instants/s : 0
Clips impossibles à 12,5 instants/s : 2707


=== fr-CA — instants CTC nécessaires par seconde
count    8077.000000
mean       11.703781
std         2.538762
min         1.048355
25%        10.030608
50%        11.779234
75%        13.420567
max        22.596974
Name: ctc_per_sec, dtype: float64
Clips impossibles à 25 instants/s : 0
Clips impossibles à 12,5 instants/s : 3136


**Résultat** : médiane ≈ 12 instants/s, maximum 22,6. Avec une analyse à 100 instants/s : réduction **÷8 exclue** (44 % des clips impossibles), **÷4 possible mais marge faible**, ÷2 confortable.

**Décision** reportée à la conception de l'architecture (choix entre ÷2 et ÷4).

## Point 6 — Répartition des locuteurs

Objectif : savoir si quelques personnes dominent les données (le modèle s'habituerait surtout à leur voix).

In [8]:
for name, df in [("fr-FR", train_fr), ("fr-CA", train_ca)]:
    # Nombre de clips enregistrés par chaque locuteur, du plus actif au moins actif
    clips_per_speaker = df["client_id"].value_counts()

    print(f"=== {name}")
    print("Locuteurs différents :", len(clips_per_speaker))
    print("Clips par locuteur :")
    print(clips_per_speaker.describe())

    # Part des clips enregistrés par les 10 locuteurs les plus actifs
    top10_share = clips_per_speaker.head(10).sum() / len(df) * 100
    print(f"Part des 10 locuteurs les plus actifs : {top10_share:.1f} %")

=== fr-FR
Locuteurs différents : 106
Clips par locuteur :
count    106.000000
mean      47.981132
std       86.058339
min        1.000000
25%        5.000000
50%       14.500000
75%       37.000000
max      505.000000
Name: count, dtype: float64
Part des 10 locuteurs les plus actifs : 53.8 %
=== fr-CA
Locuteurs différents : 169
Clips par locuteur :
count    169.000000
mean      47.792899
std       98.846374
min        1.000000
25%        8.000000
50%       18.000000
75%       44.000000
max      909.000000
Name: count, dtype: float64
Part des 10 locuteurs les plus actifs : 43.6 %


**Résultat** : 106 locuteurs (fr-FR) et 169 (fr-CA) ; les 10 plus actifs font 53,8 % et 43,6 % des clips.

**Décision** : tout garder pour le prototype (option A). Si l'écart de WER train/dev est grand : échantillonnage pondéré (option D).

## Point 7 — Dataset spontané : fichiers, qualité, normalisation

In [9]:
import os

SPONT_DIR = "data/sps-corpus-5.0-2026-09-11-fr"

# 1. Comparer les mp3 du dossier avec ceux du tableau principal
mp3_in_folder = set(os.listdir(f"{SPONT_DIR}/audios"))
mp3_in_table = set(spont["audio_file"])
extra = mp3_in_folder - mp3_in_table   # présents dans le dossier mais pas dans le tableau
print("mp3 dans le dossier :", len(mp3_in_folder))
print("mp3 dans le tableau :", len(mp3_in_table))
print("mp3 en trop :", len(extra))

# Ces mp3 en trop sont-ils les enregistrements signalés ?
reported = pd.read_csv(f"{SPONT_DIR}/ss-reported-audios-fr.tsv", sep="	")
print("Enregistrements signalés :", len(reported))
print("mp3 en trop qui sont signalés :", len(extra & set(reported["audio_file"])))
print("Raisons des signalements :")
print(reported["reason"].value_counts())

# 2. Étiquettes de qualité des clips transcrits
print(spont_ok["quality_tags"].value_counts(dropna=False))

# 3. Notre normalisation appliquée aux transcriptions spontanées
spont_text = spont_ok["transcription"].map(normalize)
print("Transcriptions écartées par la normalisation :", (~spont_text.map(is_valid)).sum())

mp3 dans le dossier : 716
mp3 dans le tableau : 688
mp3 en trop : 28
Enregistrements signalés : 28
mp3 en trop qui sont signalés : 28
Raisons des signalements :
reason
different_language    27
other                  1
Name: count, dtype: int64
quality_tags
NaN                                 245
transcription-length                 47
short-audio|transcription-length      2
Name: count, dtype: int64
Transcriptions écartées par la normalisation : 54


**Résultat** : les 28 mp3 en trop = les 28 enregistrements signalés. 49 clips étiquetés `transcription-length`. Transcriptions avec annotations entre crochets, chiffres et « euh ». Clips jusqu'à 83 s.

**Règles d'évaluation décidées** : écarter `transcription-length`, `[unclear]`/`[pas clair]` et chiffres ; supprimer `[noise]`, `[toux]`, `[disfluency]`, `[no sound]` et les « euh » ; garder les clips longs et calculer le WER par tranche de durée.

### Jeu de test spontané final

In [10]:
# Étiquettes de bruit à supprimer du texte (ce ne sont pas des mots)
NOISE_TAGS = r"\[(?:noise|toux|disfluency|no sound)\]"
# Hésitations : "euh", "euhh"... avec la virgule qui suit éventuellement
FILLERS = r"euh+,?"

def clean_spontaneous(text):
    text = re.sub(NOISE_TAGS, " ", text, flags=re.IGNORECASE)
    text = re.sub(FILLERS, " ", text, flags=re.IGNORECASE)
    # Puis la même normalisation que pour l'entraînement
    return normalize(text)

# Règle 1 : écarter les clips dont la transcription est signalée comme suspecte
spont_test = spont_ok[spont_ok["quality_tags"].isna()].copy()

# Règles 2 et 3 : écarter les passages incompréhensibles et les chiffres
has_unclear = spont_test["transcription"].str.contains(r"\[(?:unclear|pas clair)\]", case=False)
has_digit = spont_test["transcription"].str.contains(r"\d")
spont_test = spont_test[~has_unclear & ~has_digit].copy()

# Règles 4 et 5 : nettoyer le texte (bruits et "euh"), puis normaliser
spont_test["text"] = spont_test["transcription"].map(clean_spontaneous)

# Sécurité : écarter ce qui contient encore un caractère non autorisé
spont_test = spont_test[spont_test["text"].map(is_valid)]

print("Clips de test spontanés :", len(spont_test))
print("Locuteurs :", spont_test["client_id"].nunique())
print("Durée totale :", round(spont_test["duration_ms"].sum() / 60000, 1), "minutes")

# Règle 6 : répartition par tranche de durée (pour le WER par tranche)
duration_s = spont_test["duration_ms"] / 1000
print(pd.cut(duration_s, bins=[0, 11, 30, 1000]).value_counts().sort_index())

Clips de test spontanés : 230
Locuteurs : 38
Durée totale : 50.8 minutes
duration_ms
(0, 11]       131
(11, 30]       79
(30, 1000]     20
Name: count, dtype: int64
